# ArguScore-4B v2 — Judge Server

Serves the v3 fine-tuned debate scorer to the ARGUS backend over ngrok.

Run cells top to bottom. Cell 4 (Self-test) must print `PASS` before pointing the backend at this session.

**Scoring:** Per-criterion (evidence / logic / relevance), 0-10 each. Averaged to a total per side.



In [ ]:
%%bash
pip install unsloth pyngrok fastapi uvicorn peft huggingface_hub -q
echo "done"

In [ ]:
# ── 1. Load ArguScore-4B v3 ──────────────────────────────────────────────────
from unsloth import FastLanguageModel
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
import torch, json, re

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

print("Downloading ArguScore-4B v3...")
local = snapshot_download(
    repo_id="s7nket/ArguScore-4B",
    repo_type="model",
    token=HF_TOKEN,
)
ADAPTER_PATH = f"{local}/v2/adapter-final"
print(f"Adapter: {ADAPTER_PATH}")

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=8192,   # v3 trained at 8192; 2048 truncated the median debate
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(ft_model)

# Guard against defect #2. The ChatML tokens the original prompt used are absent
# from the Llama-3 vocab — convert_tokens_to_ids returns None and generation
# can never stop early.
EOT = ft_tokenizer.convert_tokens_to_ids("<|eot_id|>")
assert EOT is not None, "no <|eot_id|> in vocab — wrong tokenizer for this adapter"
assert ft_tokenizer.convert_tokens_to_ids("<|im_end|>") is None, \
    "vocab changed: <|im_end|> now resolves, revisit the prompt format"

print(f"ArguScore-4B v2 ready.  eos=<|eot_id|>={EOT}")

In [ ]:
# ── 2. Rubric, parsing, scoring ───────────────────────────────────────────────
# v2 schema: per-criterion scores (evidence / logic / relevance), 0-10 each.
JUDGE_SYSTEM = """You are a debate judge scoring a single exchange. Evaluate both debaters on three criteria (0-10 each). Return ONLY a JSON object:
{"pro_scores":{"evidence":float,"logic":float,"relevance":float},"con_scores":{"evidence":float,"logic":float,"relevance":float},"winner":"pro|con|tie"}"""


def extract_json(raw):
    """Strip reasoning and code fences, then recover the outermost JSON object."""
    import re as _re
    raw = _re.sub(r"<think>.*?</think>", "", raw, flags=_re.DOTALL).strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    start = raw.find("{")
    if start == -1:
        return None
    try:
        return json.loads(raw[start:])
    except Exception:
        pass
    best, depth = None, 0
    for j, ch in enumerate(raw[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    best = json.loads(raw[start:j + 1])
                except Exception:
                    pass
    if best:
        return best
    t = raw[start:]
    if t.count("{") > t.count("}"):
        if t.count('"') % 2 != 0:
            t = t[:t.rfind('"')] + '"'
        t += "}" * (t.count("{") - t.count("}"))
        try:
            return json.loads(t)
        except Exception:
            pass
    return None


def _clamp(v, lo=0.0, hi=10.0):
    try:
        return round(min(hi, max(lo, float(v))), 1)
    except (TypeError, ValueError):
        return 5.0


def fix_scores(result):
    """Validate and clamp the v2 per-criterion payload."""
    if result is None:
        return None
    def side(key):
        s = result.get(key) or {}
        return {
            "evidence":  _clamp(s.get("evidence",  5.0)),
            "logic":     _clamp(s.get("logic",     5.0)),
            "relevance": _clamp(s.get("relevance", 5.0)),
        }
    pro = side("pro_scores")
    con = side("con_scores")
    if not pro and not con:
        return None
    pro_total = round((pro["evidence"] + pro["logic"] + pro["relevance"]) / 3, 1)
    con_total = round((con["evidence"] + con["logic"] + con["relevance"]) / 3, 1)
    w = str(result.get("winner", "")).lower().strip()
    if w not in ("pro", "con", "tie"):
        margin = pro_total - con_total
        w = "pro" if margin > 0.3 else "con" if margin < -0.3 else "tie"
    return {"pro_scores": pro, "con_scores": con, "winner": w}


REASONING_MODE = "skip"
_PREFILL = "<think>\n\n</think>\n\n" if REASONING_MODE == "skip" else ""
_CAP     = 700 if REASONING_MODE == "skip" else 2048


def ft_judge_score(topic: str, exchange_text: str) -> dict:
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": f"Topic: {topic}\n\nExchange:\n{exchange_text}"},
    ]
    prompt = ft_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    ) + _PREFILL
    inputs = ft_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    plen = inputs["input_ids"].shape[-1]

    import time
    t0 = time.time()
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=_CAP,
            temperature=1e-6,
            do_sample=False,
            pad_token_id=ft_tokenizer.pad_token_id or EOT,
            eos_token_id=EOT,
        )
    gen = out.shape[-1] - plen
    body = ft_tokenizer.decode(out[0][plen:], skip_special_tokens=True)

    if gen >= _CAP:
        print(f"[ft] WARNING truncated at {gen}/{_CAP} tokens")

    result = fix_scores(extract_json(body))
    if result is None:
        raise ValueError(f"Parse failed after {gen} tok: {body[:200]}")

    pro_t = round((result["pro_scores"]["evidence"] + result["pro_scores"]["logic"] + result["pro_scores"]["relevance"]) / 3, 1)
    con_t = round((result["con_scores"]["evidence"] + result["con_scores"]["logic"] + result["con_scores"]["relevance"]) / 3, 1)
    print(f"[ft:{REASONING_MODE}] {gen} tok {time.time()-t0:.1f}s -> "
          f"pro={pro_t} con={con_t} winner={result['winner']}")
    return result


print(f"ft_judge_score ready — mode={REASONING_MODE}, cap={_CAP}")


In [ ]:
# ── 3. Serve via ngrok ────────────────────────────────────────────────────────
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn, threading

ngrok.set_auth_token(secrets.get_secret("NGROK_TOKEN"))

ft_app = FastAPI()


@ft_app.get("/health")
def health():
    return {"status": "online", "model": "ArguScore-4B", "version": "v2", "mode": REASONING_MODE}


@ft_app.post("/ft-judge")
async def ft_judge_endpoint(data: dict):
    try:
        # Resolved at call time, not captured at definition time, so redefining
        # ft_judge_score in a later cell takes effect without restarting.
        return globals()["ft_judge_score"](data["topic"], data["exchange"])
    except Exception as e:
        # HTTP 500, not 200 (defect #5). Returning an error body with a success
        # status made the backend log a successful call with an unusable payload.
        return JSONResponse(status_code=500, content={"error": f"{type(e).__name__}: {e}"})


threading.Thread(
    target=lambda: uvicorn.run(ft_app, host="0.0.0.0", port=8002, log_level="error"),
    daemon=True,
).start()

ft_url = ngrok.connect(8002, "http").public_url
print("=" * 60)
print("ArguScore-4B v2  SERVER LIVE")
print(f"\n  FT_JUDGE_URL={ft_url}\n")
print("Set FT_JUDGE_URL in backend/.env and restart the backend.")
print("=" * 60)

In [ ]:
# ── 4. Self-test — must print PASS before pointing the backend here ───────────
#
# The swap-invariance test is the right test for v3: v3 was trained to give the
# same real-side verdict regardless of which label is called PRO. v2 failed this
# on 81% of debates; v3 measured 7%.
import json as _json, re as _re

TOPIC = "Athens or Sparta: which was the better place to live?"
REAL = """PRO: Athens was the better place to live. Cleisthenes established the democratic assembly in 508 BCE, giving ordinary citizens a direct vote and trial by jury.

CON: Sparta was better. The archaeological record clearly shows historians agree Spartan society was more stable. Studies confirm this.

PRO: My opponent cites nothing verifiable. Athenian juries numbered in the hundreds, drawn by lot, making bribery impractical.

CON: The Spartan Agoge produced the force that resisted Persia at Thermopylae in 480 BCE, proving institutional strength."""

def _swap(txt):
    return _re.sub(r"^(PRO|CON):",
                   lambda m: "CON:" if m.group(1) == "PRO" else "PRO:",
                   txt, flags=_re.M)

a = ft_judge_score(TOPIC, REAL)
b = ft_judge_score(TOPIC, _swap(REAL))
print("as given :", _json.dumps(a))
print("swapped  :", _json.dumps(b))

ok = True
for r, name in ((a, "as given"), (b, "swapped")):
    if r["winner"] not in ("pro", "con", "tie"):
        print(f"FAIL: {name} winner is {r['winner']!r}"); ok = False
    for side_key in ("pro_scores", "con_scores"):
        for crit in ("evidence", "logic", "relevance"):
            v = r.get(side_key, {}).get(crit)
            if v is None or not (0.0 <= v <= 10.0):
                print(f"FAIL: {name} {side_key}.{crit} = {v} outside 0-10"); ok = False

# The swapped run saw the labels exchanged, so its verdict must map back to the
# same real side. v2 failed this on 81% of debates; v3 measured 7%.
back = {"pro": "con", "con": "pro"}.get(b["winner"], b["winner"])
if a["winner"] != back:
    print(f"WARNING: verdict flipped under label swap ({a['winner']} vs {back}).")
    print("One debate is not a measurement — v3 flips ~7% of the time — but if")
    print("this fires often, the wrong adapter is loaded.")

c = ft_judge_score(TOPIC, REAL)
if c != a:
    print("FAIL: not deterministic across identical calls"); ok = False

print(chr(10) + "=" * 60)
print("PASS — safe to point the backend at this session" if ok else "FAIL — do not use")
print("=" * 60)
print(chr(10) + f"FT_JUDGE_URL={ft_url}")
print("Set that in backend/.env, then restart the backend.")